# Analyze Traces with SQL and Litefuse

**Module 7: Trace Analysis and Litefuse** · [Lesson](course.md) · [Quiz](quiz07_analysis_and_litefuse.ipynb)

In this Lab you analyze traces in two ways. First you measure the payment incident with SQL on the spans that the demo's services write to `otel_traces`: the failure rate and the latency of each operation, how the failed orders differ from the others, the service in which every failure starts, and the share of failed orders over time. Then you trace a small application built on a large language model (LLM) with Litefuse, an LLM observability platform that keeps its data in Doris. You send the application's traces with the Langfuse Python SDK, find them in Doris with SQL, and open the same traces in Litefuse's web UI.

The application calls a simulated model in the notebook: it needs no API key or account, and nothing leaves your computer.

Plan about 50 minutes: 20 for sections 1 and 2, 30 for sections 3 to 9. You need the DOG stack and the demo from Module 1, with the payment failure turned on (section 5 of Lab 1). Section 4 starts Litefuse; its first start downloads about 1.2 GB of images, which takes a few minutes more.

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


## 1. Check the spans in Doris

The queries below rank and summarize server spans. A server span covers the work that a service does for one incoming request; its `span_kind` is `SPAN_KIND_SERVER`. Count them by service:

In [ ]:
server_spans = lab.wait_for_rows(
    """
    SELECT service_name, COUNT(*) AS server_spans, MAX(timestamp) AS latest_span
    FROM otel_traces
    WHERE span_kind = 'SPAN_KIND_SERVER'
    GROUP BY service_name
    ORDER BY server_spans DESC
    """,
    title="Server spans by service in otel_traces",
    hint="otel_traces has no server spans. Start the demo with lab.start_demo() in Module 1, "
         "wait a minute, then rerun this cell.",
)

**Expected result**

One row per service that handles requests, about 14 once the demo has run for a few minutes. `frontend-proxy` and `frontend` have the most server spans; `checkout`, `payment`, and `email` have few, because the load generator browses far more often than it places orders. The span that you sent in Module 2 is not listed: it is an internal span, not a server span.

## 2. Measure the incident with SQL

In Module 3 you followed the incident one request at a time. SQL on `otel_traces` measures all of it at once. The queries of this section ask about the incident window, from the first failed order until now. Find its start, then the server operations that had failed requests in it: for each, how many requests it handled, how many and what share of them failed, and how long they took.

In [ ]:
incident = lab.wait_for_rows(
    """
    SELECT MIN(timestamp) AS first_failed_order, COUNT(*) AS failed_orders
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
      AND status_code = 'STATUS_CODE_ERROR'
    HAVING COUNT(*) > 0
    """,
    title="The failed orders in otel_traces",
    hint="No order has failed. Check that the demo runs (lab.start_demo() in Module 1) and that paymentFailure "
         "is 50% (section 5 of Lab 1), wait a minute or two, then rerun this cell.",
    timeout_seconds=180,
)
INCIDENT_START = str(incident["first_failed_order"][0])

lab.sql(f"""
SELECT service_name, span_name,
       COUNT(*) AS requests,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed,
       ROUND(100 * SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) / COUNT(*), 1) AS failed_percent,
       ROUND(PERCENTILE_APPROX(duration, 0.5) / 1000.0, 2) AS p50_ms,
       ROUND(PERCENTILE_APPROX(duration, 0.99) / 1000.0, 2) AS p99_ms
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER'
  AND timestamp >= '{INCIDENT_START}'
GROUP BY service_name, span_name
HAVING SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) > 0
ORDER BY failed_percent DESC
""", title="Server operations with failed requests since the first failed order");

**Expected result**

First the start of the incident window and the number of failed orders. Then usually four rows:

- `checkout`'s `oteldemo.CheckoutService/PlaceOrder` and `payment`'s `grpc.oteldemo.PaymentService/Charge`, with the same numbers of requests and of failures, about half of them; while there are only a few orders, chance can put the share far from half. Every order makes one charge, and a failed charge fails its order.
- `frontend`'s `POST`, with the same failures among more requests: the storefront's other POST requests, such as adding a product to the cart, carry the same span name.
- `frontend-proxy`'s `ingress`, with the same failures among tens of thousands of requests: a failure rate of almost 0. A gateway's rate hides the failure of one operation among all the others; break the rate down by operation.

`p50_ms` is the median duration and `p99_ms` the 99th percentile: 99% of the requests took at most that long. `PERCENTILE_APPROX` computes them from a compact summary of the durations instead of sorting them all, which is accurate enough for latency. The `duration` column holds microseconds, so the query divides it by 1,000. With a few dozen orders, a single slow order sets the P99 of `PlaceOrder`; a percentile means more with more requests.

### Failed orders and successful orders

Compare the duration of the orders that failed with that of the orders that succeeded:

In [ ]:
lab.sql(f"""
SELECT IF(status_code = 'STATUS_CODE_ERROR', 'failed', 'succeeded') AS outcome,
       COUNT(*) AS orders,
       ROUND(PERCENTILE_APPROX(duration, 0.5) / 1000.0, 1) AS p50_ms,
       ROUND(PERCENTILE_APPROX(duration, 0.99) / 1000.0, 1) AS p99_ms,
       ROUND(MAX(duration) / 1000.0, 1) AS max_ms
FROM otel_traces
WHERE service_name = 'checkout'
  AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
  AND timestamp >= '{INCIDENT_START}'
GROUP BY outcome
""", title="The duration of the orders since the first failed order, by outcome");

**Expected result**

Two rows. The failed orders are faster: their median is well below that of the successful orders, for example 6 ms against 11 ms. An order fails at the charge, while a successful order goes on: the checkout service ships it, empties the cart, sends the confirmation email, and publishes the order to Kafka. A failure does not always make requests slower; an alert on latency alone would not see this incident.

### Where each failure starts

In Module 3 you found the start of one failure in its trace: the deepest span with an error. To find it for every failed order with SQL, compare the services of a trace. A service that received a request is where the failure starts when its server span failed while none of the calls that it made to other services failed: it did not fail because of another service. The next query finds such server spans in the traces of the failed orders, with `NOT EXISTS`, which keeps a row only when the subquery finds no matching row for it. It leaves out the orders of the last minute, whose spans may not all have reached Doris.

In [ ]:
lab.sql(f"""
SELECT s.service_name, s.span_name, COUNT(DISTINCT s.trace_id) AS failed_orders
FROM otel_traces AS s
WHERE s.span_kind = 'SPAN_KIND_SERVER'
  AND s.status_code = 'STATUS_CODE_ERROR'
  AND s.timestamp >= '{INCIDENT_START}'
  AND s.trace_id IN (
      SELECT trace_id
      FROM otel_traces
      WHERE service_name = 'checkout'
        AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
        AND status_code = 'STATUS_CODE_ERROR'
        AND timestamp >= '{INCIDENT_START}'
        AND end_time < UTC_TIMESTAMP() - INTERVAL 1 MINUTE
  )
  AND NOT EXISTS (
      SELECT 1
      FROM otel_traces AS c
      WHERE c.trace_id = s.trace_id
        AND c.service_name = s.service_name
        AND c.span_kind = 'SPAN_KIND_CLIENT'
        AND c.status_code = 'STATUS_CODE_ERROR'
  )
GROUP BY s.service_name, s.span_name
""", title="Server spans that failed without a failed call, in the traces of the failed orders");

**Expected result**

One row: `payment`, `grpc.oteldemo.PaymentService/Charge`, with every failed order older than a minute. Each of the other services on the path, `checkout`, `frontend`, and `frontend-proxy`, has a failed call of its own in the trace, so the subquery finds a row, and `NOT EXISTS` leaves it out: they failed because the payment service did.

The query compares services, not single spans, on purpose. In the frontend, the span `POST /api/checkout` sits between two error spans but does not record an error itself, so a rule that looks for an error span whose children have no error stops at the frontend's `POST` too. And a trace's spans come from different services, whose clocks and SDKs can disagree by a fraction of a millisecond, so the span that ended first is not always the one that failed first.

### The share of failed orders over time

Lab 1 counted the orders per minute and noted that, with one to three orders a minute, a single minute says little. A window function can compute the share of failures over the last ten orders, for every order. Unlike `GROUP BY`, a window function keeps every row and computes its value over a set of rows related to it, the window: here, the orders in time order, and for `AVG`, a frame of the order and the nine before it.

In [ ]:
lab.sql("""
SELECT order_number, ordered_at, outcome, failed_share_of_last_10
FROM (
    SELECT ROW_NUMBER() OVER (ORDER BY timestamp) AS order_number,
           timestamp AS ordered_at,
           IF(status_code = 'STATUS_CODE_ERROR', 'failed', 'succeeded') AS outcome,
           ROUND(AVG(IF(status_code = 'STATUS_CODE_ERROR', 1, 0))
                 OVER (ORDER BY timestamp ROWS BETWEEN 9 PRECEDING AND CURRENT ROW), 2) AS failed_share_of_last_10
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
) AS orders
ORDER BY order_number DESC
LIMIT 15
""", title="The latest orders, with the share of failures among the last ten");

**Expected result**

Up to 15 rows, the latest order first. `ROW_NUMBER()` numbers all of the orders since the demo started, in time order, so the latest order has the highest number. `failed_share_of_last_10` stays around 0.5, between about 0.3 and 0.7, while single orders alternate between `failed` and `succeeded` at random. For the first orders after you turned on the flag, the frame still holds orders from before it, and the share is lower; while the demo has placed only a few orders, the frame holds only those, and chance can put the share far from 0.5. `PARTITION BY` would start the numbering and the frames again for each group, such as each service; Module 8 alerts on a share like this one.

## 3. Litefuse: traces of an LLM application

The spans in `otel_traces` describe requests between services: which service called which, and how long each call took. An application built on an LLM needs more about each request: the prompt that it sent to the model, the model's answer, how many tokens the model counted and what they cost, and the steps around the model call, such as the search for documents to put in the prompt. LLM observability platforms record a trace for each request too, with an observation for each step.

Litefuse is such a platform. It is an open-source fork of Langfuse ([github.com/litefuse/litefuse](https://github.com/litefuse/litefuse), MIT license): Langfuse keeps its traces in ClickHouse, Litefuse keeps them in Apache Doris. Its API is compatible with Langfuse's, so applications send their traces with Langfuse's SDKs, pointed at Litefuse's address. In this course, Litefuse keeps its tables in the DOG stack's Doris, in the database `litefuse`, next to `otel`.

Litefuse 26.3.0 runs as five containers:

| Container | What it does |
| --- | --- |
| `doris-obs-litefuse-web` | The web UI, and the API that receives the traces |
| `doris-obs-litefuse-worker` | Loads the traces into Doris in the background |
| `doris-obs-litefuse-postgres` | PostgreSQL: users, organizations, projects, API keys, and prompts |
| `doris-obs-litefuse-redis` | Redis: the queues between the web container and the worker |
| `doris-obs-litefuse-silo` | Silo, an S3-compatible object storage server forked from MinIO: each batch of traces that the API receives |

The API keeps each batch that it receives in object storage and queues it. The worker loads the queued batches into Doris with Stream Load, and the UI and the API read the traces back from Doris with SQL.

## 4. Start Litefuse

`lab.start_litefuse()` starts Litefuse with the Compose file `environments/litefuse/compose.litefuse.yml`, which pins the version of every image. Litefuse starts with a user, an organization `Doris course`, and a project `LLM app` with a pair of API keys, so you do not sign up.

Before it starts the containers, the cell runs one statement on Doris, so that Litefuse can load data:

- Litefuse loads its data with Stream Load through the FE, at `doris-obs:8030`. The FE does not take the data itself: it answers with a redirect to a BE, at the address that the BE registered with.
- The BE of this single-container Doris registered as `127.0.0.1`. From another container, that address is the container itself, and the load fails with "connection refused".
- `ALTER SYSTEM MODIFY BACKEND` gives the BE a private endpoint, the tag `private_endpoint`. When a BE has one, the FE redirects a Stream Load to the host that the client used to reach the FE, here `doris-obs`, with the port of the endpoint, 8040. Containers on the `doris-obs` network reach the BE there. [Stream Load Practices in Complex Network Environments](https://doris.apache.org/docs/4.x/data-operate/import/load-best-practices/stream-load-in-complex-network/) describes the endpoints.

In [ ]:
lab.start_litefuse()

**Expected result**

In order:

- The port check: 23100 (`DORIS_OBS_LITEFUSE_PORT`) is `available`, or `used by this course` when Litefuse already runs.
- The statement `ALTER SYSTEM MODIFY BACKEND "127.0.0.1:9050"` with the tags `location` and `private_endpoint`, and "Statement succeeded". `SHOW BACKENDS` now shows both tags in its `Tag` column.
- The log of `docker compose`: on the first run, the downloads, then five `doris-obs-litefuse-*` containers, which `ps` lists as running; Postgres, Redis, and Silo also as healthy.
- The project that the API keys belong to: the organization `Doris course`, the project `LLM app`, and its ID, `llm_app`.
- The card "Litefuse is running", about 10 seconds after the containers start: on its first start, Litefuse creates its database in Postgres and its tables in Doris before it answers.

Rerunning the cell keeps the running containers and their data.

## 5. Trace a small LLM application

The next cell builds a small assistant that answers questions about this course, and traces it with the Langfuse Python SDK (`langfuse`, installed with the course's requirements). Each step of the assistant is a function with the `@observe` decorator, which records each call as an observation, with the function's arguments as its input and its return value as its output:

- `answer_question` handles one request and is the root of its trace. `propagate_attributes` names the trace `course-assistant` and records the user and the session that asked; the observations below it inherit them.
- `search_docs` looks up the documents about the topic of the question: an observation of type `retriever`.
- `call_model` stands for the call to the model: an observation of type `generation`. The simulated model returns a prepared answer, counts one token per word, and writes 100 tokens per second. `update_current_generation` records the prompt (the chat messages), the model, the token counts, and the cost of the call, from an example price of 0.01 USD per 1,000 input tokens and 0.03 USD per 1,000 output tokens. With a real model, Langfuse's integrations, such as the one for the OpenAI SDK, record these by themselves: [Optional Lab 7A](optional07_trace_a_real_model.ipynb) gives this assistant a real model, on your computer or OpenAI's.

The SDK reads the API keys and Litefuse's address from the variables `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and `LANGFUSE_BASE_URL`. It sends the observations as OpenTelemetry spans, in batches, to Litefuse's endpoint `/api/public/otel/v1/traces`; `flush()` sends the rest before the cell ends. `OTEL_SERVICE_NAME` names the application in the spans' resource, as the demo's services are named in `otel_traces`.

In [ ]:
import os
import time

os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-doris-course"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-doris-course"
os.environ["LANGFUSE_BASE_URL"] = lab.litefuse_url
os.environ["OTEL_SERVICE_NAME"] = "course-assistant"

from langfuse import get_client, observe, propagate_attributes

langfuse = get_client()

# The documents that the assistant searches, by topic, and the answers of the simulated model.
DOCUMENTS = {
    "doris": "Apache Doris is a real-time analytical database. It answers SQL queries over large tables in seconds.",
    "litefuse": "Litefuse is an LLM engineering platform. It keeps its traces and observations in Apache Doris tables.",
    "collector": "The OpenTelemetry Collector receives logs, metrics, and traces and exports them to Doris.",
}
ANSWERS = {
    "What is Apache Doris?":
        "Apache Doris is a real-time analytical database that answers SQL queries over large tables in seconds.",
    "Where does Litefuse keep its traces?":
        "Litefuse keeps its traces and observations in Apache Doris tables, so you can query them with SQL.",
}
PRICE_PER_1K_TOKENS = {"input": 0.01, "output": 0.03}  # an example price list, in USD


def count_tokens(text):
    return len(text.split())  # the simulated model counts one token per word


@observe(as_type="retriever", name="search-docs")
def search_docs(question):
    time.sleep(0.05)  # a lookup in a document index takes a moment
    return [text for topic, text in DOCUMENTS.items() if topic in question.lower()]


@observe(as_type="generation", name="llm-answer")
def call_model(question, documents):
    messages = [
        {"role": "system", "content": "Answer the question with these documents: " + " ".join(documents)},
        {"role": "user", "content": question},
    ]
    answer = ANSWERS[question]
    usage = {"input": sum(count_tokens(message["content"]) for message in messages),
             "output": count_tokens(answer)}
    time.sleep(usage["output"] / 100)  # the simulated model writes 100 tokens per second
    langfuse.update_current_generation(
        input=messages,
        model="simulated-llm",
        model_parameters={"temperature": 0},
        usage_details=usage,
        cost_details={kind: round(tokens / 1000 * PRICE_PER_1K_TOKENS[kind], 6) for kind, tokens in usage.items()},
    )
    return answer


@observe(name="answer-question")
def answer_question(question, *, user_id, session_id):
    with propagate_attributes(trace_name="course-assistant", user_id=user_id, session_id=session_id):
        trace_ids.append(langfuse.get_current_trace_id())
        return call_model(question, search_docs(question))


trace_ids = []
for user_id, session_id, question in [
    ("user-1", "session-1", "What is Apache Doris?"),
    ("user-1", "session-1", "Where does Litefuse keep its traces?"),
    ("user-2", "session-2", "What is Apache Doris?"),
]:
    print(f"{user_id}: {question}\n  -> {answer_question(question, user_id=user_id, session_id=session_id)}")
langfuse.flush()
print("Trace IDs:", trace_ids)

**Expected result**

Three questions with their answers, two from `user-1` in `session-1` and one from `user-2` in `session-2`, then `Trace IDs:` and three IDs of 32 hexadecimal digits: OpenTelemetry trace IDs, like those in `otel_traces`. The next sections look up these three traces. Running the cell again sends three new traces, and the next sections then show the new ones.

## 6. Find the traces in Doris

When the first trace of a project arrives, Litefuse creates the project's two tables and names them after the project's ID, `llm_app`: `spans_llm_app` has a row per observation, and `traces_scalar_llm_app` a row per trace. The next cell waits for them.

In [ ]:
lab.wait_for_rows(
    """
    SELECT TABLE_NAME AS table_name, TABLE_TYPE AS table_type
    FROM information_schema.tables
    WHERE TABLE_SCHEMA = 'litefuse' AND TABLE_NAME LIKE '%llm_app'
    ORDER BY TABLE_NAME
    """,
    title="The tables of the project llm_app",
    hint="Litefuse creates them when the project's first trace arrives. Check that the previous cell printed "
         "three trace IDs and that Litefuse runs (section 4). The worker's log, docker logs "
         "doris-obs-litefuse-worker, shows the creation of the tables. Then rerun this cell.",
    min_rows=2,
);

**Expected result**

Two rows, `spans_llm_app` and `traces_scalar_llm_app`, both `BASE TABLE`, within a few seconds of the first traces.

In [ ]:
lab.sql("SHOW TABLES FROM litefuse", title="Litefuse's tables in Doris");

**Expected result**

Eight tables. Besides the project's two, Litefuse created six when it first started: `scores` holds scores of traces, such as user feedback or the results of evaluations; `dataset_run_items_rmt` the results of experiments on datasets; `event_log` and `blob_storage_file_log` where in object storage each event that the API received is kept; `project_environments` the environments of each project; and `schema_migrations` the migrations that Litefuse has applied to the database. Each project gets its own pair of tables for its spans and traces.

The next cell waits for the observations of the three traces. `usage_details` is a `MAP` of token counts by kind, and `TIMESTAMPDIFF` gives each observation's duration from its start and end times.

In [ ]:
TRACE_ID_LIST = ", ".join(f"'{trace_id}'" for trace_id in trace_ids)
llm_spans = lab.wait_for_rows(
    f"""
    SELECT trace_id, user_id, name, type, provided_model_name AS model, usage_details, total_cost,
           TIMESTAMPDIFF(MICROSECOND, start_time, end_time) / 1000 AS duration_ms
    FROM litefuse.spans_llm_app
    WHERE trace_id IN ({TRACE_ID_LIST})
    ORDER BY start_time, is_root DESC
    """,
    title="The observations of the three traces in spans_llm_app",
    hint="Litefuse loads a batch of traces into Doris a few seconds after it receives it. Check the worker's "
         "log with docker logs doris-obs-litefuse-worker, then rerun this cell.",
    min_rows=9,
    timeout_seconds=90,
)

**Expected result**

Nine rows, three per trace, in the order of the requests. The first time, they arrive about 10 seconds after the traces were sent: Litefuse's worker waits that long before it loads data into a project's new tables. Later runs take a few seconds. For a question about Doris:

| `name` | `type` | `model` | `usage_details` | `total_cost` | `duration_ms` |
| --- | --- | --- | --- | --- | --- |
| `answer-question` | `SPAN` | NULL | `{}` | NULL | about 220 |
| `search-docs` | `RETRIEVER` | NULL | `{}` | NULL | about 55 |
| `llm-answer` | `GENERATION` | `simulated-llm` | `{"input":26, "output":16, "total":42}` | `0.000740000000` | about 160 |

The question about Litefuse has `{"input":28, "output":17, "total":45}` and `0.000790000000`. Litefuse added the totals: `total` in `usage_details`, and `total_cost`. The root span of each trace takes about as long as its two steps together, and every row has the `user_id` that `propagate_attributes` set.

In [ ]:
lab.sql(f"""
SELECT input, output
FROM litefuse.spans_llm_app
WHERE trace_id = '{trace_ids[1]}' AND type = 'GENERATION'
""", title="The prompt and the answer of the second request");

**Expected result**

One row. `input` is the prompt: the two chat messages as JSON, the system message with the document about Litefuse that `search_docs` found, and the user's question. `output` is the answer. The other observations keep the arguments and the return value of their function in the same columns.

## 7. The same traces in Litefuse

Open Litefuse's web UI:

1. Go to [http://127.0.0.1:23100](http://127.0.0.1:23100) (or the port in `DORIS_OBS_LITEFUSE_PORT`) and sign in with the email `learner@example.com` and the password `doris-course-1`.
2. The page lists the organization **Doris course** with the project **LLM app**. Choose **Go to project**.
3. In the menu on the left, choose **Tracing**. The **Traces** tab lists the traces of the past day, newest first: the three traces named `course-assistant`, with their input, output, latency, and tokens.
4. Choose the trace whose input asks "Where does Litefuse keep its traces?". A panel opens with the trace as a tree: `course-assistant`, below it `answer-question`, and below that `search-docs` and `llm-answer`, each with its duration. Choose `llm-answer`: the panel shows the prompt as **System** and **User** messages, the answer as the **Assistant** message, the model `simulated-llm` with `temperature: 0`, the tokens `28 prompt → 17 completion (∑ 45)`, and the cost `$0.00079`.
5. In the menu, choose **Sessions**: `session-1` has two traces and `session-2` one; scroll the table to the right for their cost and tokens. **Users** lists the two users, and **Home** charts the traces, their cost by model and by user, and percentiles of their latency.

Litefuse reads all of this from the tables that you queried in section 6. Its public API does too: the next cell reads the second trace from it, with the same API keys as the SDK.

In [ ]:
trace = lab.litefuse_trace(trace_ids[1])

**Expected result**

The `curl` command, then a table with the trace's three observations: `SPAN` `answer-question`, `RETRIEVER` `search-docs`, and `GENERATION` `llm-answer` with the model `simulated-llm`, the usage `{'input': 28, 'output': 17, 'total': 45}`, and the cost `0.00079`. The title gives the trace's name, `course-assistant`, its latency, about 0.23 seconds, and its total cost, `0.00079` USD: the numbers that section 6 found in Doris.

## 8. How Litefuse uses Doris

Litefuse's tables use the features of Modules 4 to 6. The next cell lists the key columns of the project's tables.

In [ ]:
lab.sql("""
SELECT TABLE_NAME AS table_name, COLUMN_NAME AS key_column, COLUMN_KEY AS key_type
FROM information_schema.columns
WHERE TABLE_SCHEMA = 'litefuse'
  AND TABLE_NAME IN ('spans_llm_app', 'traces_scalar_llm_app')
  AND COLUMN_KEY <> ''
ORDER BY TABLE_NAME, ORDINAL_POSITION
""", title="Key columns of the project's tables");

**Expected result**

Five rows. `spans_llm_app` is a Duplicate Key table (`DUP`) on `trace_id`, `start_time`, and `span_id`: observations are only added, and the key keeps the rows of a trace together, in time order. `traces_scalar_llm_app` is a Unique Key table (`UNI`) on `id` and `start_time`: when Litefuse writes the row of a trace again, the new row replaces the old one, as the rows of `error_alerts` did in Module 4.

Both tables are partitioned by day with `AUTO PARTITION BY RANGE (date_trunc(start_time, 'day'))`, and distributed by the trace ID into `BUCKETS AUTO` (Module 5); `SHOW CREATE TABLE litefuse.spans_llm_app` shows these clauses. Litefuse also builds a synchronous materialized view on `spans_llm_app`, `trace_metrics_agg_llm_app`, which adds up the tokens and costs of each trace by day (Module 6); `SHOW CREATE MATERIALIZED VIEW trace_metrics_agg_llm_app ON litefuse.spans_llm_app` shows it.

`spans_llm_app` also has inverted indexes on 14 columns (`SHOW INDEX FROM litefuse.spans_llm_app` lists them). Those on `input` and `output` split the text into words (`"parser" = "unicode"`, in lowercase), so that a query can find the prompts and answers that contain a word without reading every row. The next cell searches the inputs for the word Litefuse.

In [ ]:
lab.sql("""
SELECT trace_id, name, type
FROM litefuse.spans_llm_app
WHERE input MATCH_ANY 'litefuse'
ORDER BY start_time, is_root DESC
""", title="Observations whose input contains the word litefuse");

**Expected result**

Three rows, the observations of the trace about Litefuse, for each time that you ran section 5: the question is in the input of all three, `answer-question` and `search-docs` received it as an argument, and `llm-answer` in its prompt. `MATCH_ANY` matches whole words in any case, and finds a row when any of the given words is in it; `MATCH_ALL` needs all of them.

## 9. Independent exercise

For each user, count the traces and add up the tokens and the cost of the model calls in `spans_llm_app`. The tokens of a call are in `usage_details['total']`. Expect two rows, `user-1` first.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
lab.sql("""
SELECT user_id,
       COUNT(DISTINCT trace_id) AS traces,
       SUM(usage_details['total']) AS total_tokens,
       SUM(total_cost) AS total_cost_usd
FROM litefuse.spans_llm_app
WHERE type = 'GENERATION'
GROUP BY user_id
ORDER BY total_cost_usd DESC
""", title="Traces, tokens, and cost by user");

**Expected result**

`user-1` has 2 traces, 87 tokens, and 0.00153 USD; `user-2` has 1 trace, 42 tokens, and 0.00074 USD. Each run of section 5 adds as much again. The **Sessions** page of Litefuse shows the same totals. Only the generations count: the other observations have no tokens or cost, and each trace has exactly one generation, so counting the traces among the generations counts them all.

## Completion criteria

- You can say what an LLM trace records about a request that a span in `otel_traces` does not.
- `spans_llm_app` has the observations of your three traces, with the model, the tokens, and the cost of each model call, and you found the same trace in Litefuse's web UI.
- You can explain why the BE needed a private endpoint before Litefuse could load data.
- Your query adds up the tokens and the cost by user.

## Clean up

Litefuse keeps running and uses about 1 GB of memory. `lab.stop_litefuse()` stops and removes its containers; its user, keys, and traces stay in its volumes and in Doris, and `lab.start_litefuse()` starts it again with them. `lab.teardown()` removes Litefuse with the rest of the environment, together with its volumes and the `litefuse` database.

## Next

Open [Quiz 7: Trace Analysis and Litefuse](quiz07_analysis_and_litefuse.ipynb), then continue to [Module 8: Alerting with Grafana](../module08-alerting/course.md). [Optional Lab 7A: Trace a Real Model with Litefuse](optional07_trace_a_real_model.ipynb) repeats section 5 with a real model.